# Search, evaluate, and export

Run the setup and ingest notebooks first. This notebook reloads only Drive-backed artifact fingerprints, so it works after a Colab runtime restart without re-encoding frames.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/HCM-AI-Challenge-2026')
REPO_ROOT = Path('/content/HCM-AI-Challenge-2026')
ARTIFACT_ROOT = Path(os.environ.get('ARTIFACT_ROOT', DRIVE_ROOT / 'artifacts'))
OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', DRIVE_ROOT / 'outputs'))
state_path = ARTIFACT_ROOT / 'colab_state.json'
if not state_path.is_file():
    raise FileNotFoundError(f'Run 01_ingest_index.ipynb first: {state_path}')
STATE = json.loads(state_path.read_text(encoding='utf-8'))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_cli(script, *arguments):
    command = [sys.executable, script, *map(str, arguments)]
    completed = subprocess.run(
        command, cwd=REPO_ROOT, env=os.environ.copy(), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False,
    )
    if completed.returncode:
        raise RuntimeError(f'{command} failed\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}')
    return json.loads(completed.stdout.strip().splitlines()[-1])

print(STATE)

In [ ]:
from hcm_ai.contracts import QueryRecord, TaskType
from hcm_ai.ingestion import load_aic2025_queries

AIC2025_ROOT = Path(STATE['aic2025_root'])
query_root = AIC2025_ROOT / 'query'
queries = load_aic2025_queries(query_root) if query_root.exists() else []

# Fallbacks make the smoke acceptance executable even before a query archive is mounted.
kis_query = next((item for item in queries if item.task == TaskType.KIS), QueryRecord(
    query_id='smoke_kis', text='a person in a video scene', task=TaskType.KIS
))
trake_query = next((item for item in queries if item.task == TaskType.TRAKE), QueryRecord(
    query_id='smoke_trake', text='E1: a person appears\nE2: the person moves', task=TaskType.TRAKE
))
qa_query = next((item for item in queries if item.task == TaskType.QA), QueryRecord(
    query_id='smoke_qa', text='Ai xuat hien trong video?', task=TaskType.QA
))
[(item.query_id, item.task.value, item.text[:100]) for item in (kis_query, trake_query, qa_query)]

In [ ]:
# Gemini is opt-in. The defaults below remain entirely local and leave QA answer=null with evidence.
USE_GEMINI_PLANNER = False
USE_GEMINI_ANSWERER = False

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value)

def search(record, top_k=10):
    destination = OUTPUT_ROOT / f'{safe_name(record.query_id)}.jsonl'
    args = [
        '--artifact-root', STATE['artifact_root'], '--profile', STATE['profile'],
        '--query', record.text, '--query-id', record.query_id, '--task', record.task.value,
        '--top-k', str(top_k), '--output', destination,
        '--planner', 'gemini' if USE_GEMINI_PLANNER else 'heuristic',
        '--answerer', 'gemini' if USE_GEMINI_ANSWERER else 'offline',
    ]
    for name, fingerprint in STATE['visual_indexes'].items():
        if fingerprint:
            args += ['--visual-index', f'{name}={fingerprint}']
    if STATE.get('ocr_index'):
        args += ['--ocr-index', STATE['ocr_index']]
    if STATE.get('asr_index'):
        args += ['--asr-index', STATE['asr_index']]
    return run_cli('scripts/run_retrieval.py', *args)

In [ ]:
# Colab acceptance: execute one KIS, one chronological TRAKE, and one evidence-first QA request.
kis_run = search(kis_query, top_k=10)
trake_run = search(trake_query, top_k=10)
qa_run = search(qa_query, top_k=5)
kis_run, trake_run, qa_run

In [ ]:
# Canonical JSONL is accompanied by CSV and a query trace. Validate IDs/timestamps/order/schema.
validation = {}
for name, run in {'kis': kis_run, 'trake': trake_run, 'qa': qa_run}.items():
    validation[name] = run_cli(
        'scripts/validate_submission.py', '--input', run['jsonl'], '--task', run['task']
    )
validation

In [ ]:
# Inspect audit information. Branch errors reveal safe downgrade, model-load, or quota fallbacks.
for name, run in {'kis': kis_run, 'trake': trake_run, 'qa': qa_run}.items():
    trace = json.loads(Path(run['trace']).read_text(encoding='utf-8'))
    print(name, 'results=', run['result_count'], 'branch_errors=', trace.get('branch_errors', []))
    print('plan=', trace.get('plan', {}))

In [ ]:
# Optional qualitative review of the canonical rows.
import pandas as pd
pd.read_json(kis_run['jsonl'], lines=True).head()